# Exploring an RDF dataset

## Follow chemicals through thyroid-related pathways

Use the saved schema to explore AOPWiki:

- Which chemicals appear in thyroid-related pathways?
- What names and identifiers do they have?
- Which other pathways and key events are linked to them?

## Open AOPWiki

Open the saved AOPWiki schema to get a typed Python API.
The searches below read current AOPWiki data.

The included schema was saved on 9 September 2026.

In [ ]:
from rdfsolve.client import Client

data = Client.open("../data/aopwikirdf.schema.json")

In [ ]:
data.types()

In [ ]:
from IPython.display import Markdown, display

display(Markdown(data.diagram("Adverse outcome pathway", "Stressor", "Chemical entity", "Key event")))

## Get one pathway by identifier

Already know the pathway identifier? `get()` reads that exact record, without a text search. Use its full identifier URL.

`AOP` is the generated class; `aop` is pathway 162, with its fields filled from AOPWiki.

In [ ]:
AOP = data.model("Adverse outcome pathway")
aop = data.get(AOP, "https://identifiers.org/aop/162")
print(aop.title)

Visualizing 1-hop connections for Adverse Outcome Pathway 162:

In [ ]:
nearby = data.connections(aop, max_hops=1)
display(Markdown(data.diagram(paths=nearby, instances=True)))

## Text search

`find()` searches names and identifiers. `types()` shows the classes of the matches.

In [ ]:
matches = data.find("thyroid")
matches.types()

`of_type()` selects matches by the specified type:

In [ ]:
pathways = matches.of_type("Adverse Outcome Pathway")
pathways

`paths()` lists the links between the result classes.

In [ ]:
pathways.paths()

## Which routes pass through stressors?

`via` keeps the requested intermediate class. `meaning` ranks the remaining routes using their labels.


In [ ]:
routes = data.paths_between(
    "Adverse outcome pathway", "Chemical entity",
    via=["Stressor"], meaning="stressor chemical entity", max_hops=2,
)
display(routes)
display(Markdown(data.diagram(paths=routes, path=1)))

## Which longer paths were observed?

A schema saved after path mining includes support counts. An empty table means this schema has no saved navigation profiles. `matched` describes the mining snapshot.


In [ ]:
data.navigation(max_hops=3)[["Label", "Hops", "Support", "Sources", "Matched"]]

## Follow all matching pathways

Evaluate the chemical connections for the whole thyroid selection.

In [ ]:
chemical_paths = pathways.paths_between(
    "Chemical entity", via=["Stressor"], max_hops=2,
)
chemical_paths

## Retrieve the connection and available names

Keep pathway–chemical pairs together. Missing names or identifiers stay empty.


In [ ]:
route = chemical_paths.iloc[0]["Reference"]
result = data.retrieve(
    route, source="pathway", target="chemical",
    fields={"chemical": ["title", "identifier"]},
)
result.table().head()

In [ ]:
data.query_log()

## Which fields belong to Key Events?

In [ ]:
data.describe(owners=["Key Event"], targets=["cellular organisms"])

## Follow the stressor annotations

`related()` follows links to records of the specified class.

In [ ]:
stressors = pathways.related("Stressor")
stressors

## Get the chemicals and their identifiers

`via="Stressor"` takes us through stressors to chemicals. `show()` displays the fields we ask for, alongside each record’s name and class.

In [ ]:
chemicals = pathways.related(kind="Chemical entity", via="Stressor")
chemicals.show("identifier")

## Look up a name from the list

`find()` retrieves Phenobarbital; `show()` reads its other identifiers.

In [ ]:
phenobarbital = data.find("Phenobarbital", kind="Chemical entity")
phenobarbital.show("exactmatch")

## How is this chemical connected?

First, list the pathways linked to Phenobarbital.

In [ ]:
phenobarbital_pathways = phenobarbital.related("Stressor").related("Adverse outcome pathway")
phenobarbital_pathways

`connections()` shows the actual records and links between two choices.

For the first records (```[0]```):

In [ ]:
connections = data.connections(phenobarbital[0], phenobarbital_pathways[0], max_hops=2)
display(Markdown(data.diagram(paths=connections, path=1)))

Which other pathways have these chemicals as stressors?

In [ ]:
annotations = chemicals.related("Stressor")
other_pathways = annotations.related("Adverse outcome pathway").without(pathways)
other_pathways

See the key events in those pathways

In [ ]:
events = other_pathways.related("Key event", via="has key event")
events

## Save results

Save the chemical table and the record of the session. The session keeps the schema, queries and evidence for later inspection.

In [ ]:
chemicals.show("identifier").to_csv("thyroid-chemicals.csv", index=False)
data.save_session("thyroid-session.json")
data.close()

Next: [subset RDF](02_subset.ipynb) or [parse into RDF](03_to_RDF_schema.ipynb).

In [ ]:
data.query_log()